# 🌸 Nayari Dataset Builder
**Run this notebook locally.** Reads all `.md` and `.pdf` files from `dataset/`,
builds `nayari_dataset.json`, then **auto-uploads it to Kaggle** via the API.

After upload, add the dataset to `nayari_train.ipynb` on Kaggle via **+ Add Data**.

## 📦 Step 0 — Install Libraries

In [17]:
%pip install pdfplumber kaggle -q

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.0.1 -> 26.1.1
[notice] To update, run: C:\Users\UTiaya\AppData\Local\Python\pythoncore-3.14-64\python.exe -m pip install --upgrade pip


In [10]:
import re, json, os, shutil, subprocess, sys
import pdfplumber
from pathlib import Path

DATASET_DIR = Path("dataset")
OUTPUT_FILE = Path("nayari_dataset.json")

print("Files in dataset/:")
for f in sorted(DATASET_DIR.iterdir()):
    print(f"  [{f.suffix.upper():5}] {f.name}  ({f.stat().st_size/1024:.1f} KB)")

Files in dataset/:
  [.MD  ] Aura_chat_1 (Renamed Name).md  (4.8 KB)
  [.MD  ] Aura_chat_2 (Renamed Name).md  (4.2 KB)
  [.MD  ] Aura_chat_3 (Renamed Name).md  (4.3 KB)
  [.PDF ] Discovery .pdf  (84.4 KB)
  [.PDF ] Her Beliefs .pdf  (86.0 KB)
  [.MD  ] Nayari_chat_4.md  (2.0 KB)
  [.MD  ] Nayari_chat_5.md  (5.7 KB)
  [.MD  ] Nayari_chat_6.md  (3.2 KB)
  [.MD  ] Nayari_chat_7.md  (4.7 KB)
  [.MD  ] Nayari_Details.md  (3.0 KB)


## 1 — Character Details

In [11]:
details_text = (DATASET_DIR / "Nayari_Details.md").read_text(encoding="utf-8")

def extract_field(text, field):
    m = re.search(rf"\*\*{field}\*\*:?\s*(.+)", text)
    return m.group(1).strip() if m else ""

char_name        = extract_field(details_text, "Name")
char_type        = extract_field(details_text, "Type")
char_gender      = extract_field(details_text, "Gender")
char_traits      = extract_field(details_text, "Traits")
char_personality = extract_field(details_text, "Personally")
lore_sections    = []

print(f"Character: {char_name} | {char_type} | {char_gender}")

Character: Nayari | Kemonomimi | Female (Cis)


## 2 — PDF Parser
Each page is checked: **chat page** (>=2 `Speaker:` lines) -> parsed as conversations.
**Prose/lore page** -> chunked into conversation training examples (Nayari narrates her own story).

In [ ]:
# ── Bug-fix 1: support **Bold**: speaker format used in chats 5-7
# Strip leading/trailing ** from speaker name before matching
USER_ALIASES      = {"me", "you", "tiaya"}        # added 'tiaya'
ASSISTANT_ALIASES = {"nayari", "nayri", "aura"}   # added 'nayri' typo-variant

# Matches both:  plain → Name: text
#                bold  → **Name**: text
SPEAKER_RE = re.compile(r'^\*{0,2}([A-Za-z][A-Za-z ]*?)\*{0,2}:\s*(.*)')
END_RE     = re.compile(r'^-{3,}\s*<?(end|END)>?\s*-{3,}$')


def is_chat_page(text):
    return sum(1 for l in text.splitlines() if SPEAKER_RE.match(l.strip())) >= 2


# ── Bug-fix 2: treat 2+ consecutive blank lines as an implicit <end>
# so scene breaks inside one file become separate training conversations
def _inject_scene_ends(text):
    """Replace 2+ consecutive blank lines with a synthetic END marker."""
    return re.sub(r'(
?
){3,}', '
--- END ---
', text)


def parse_chat_text(text):
    text = _inject_scene_ends(text)   # split scenes before tokenising
    lines = text.splitlines()
    conversations, current_messages = [], []
    current_role, buf = None, []

    def flush():
        nonlocal buf
        if current_role and buf:
            content = " ".join(" ".join(buf).split()).strip()
            if content:
                current_messages.append({"role": current_role, "content": content})
        buf = []

    def save():
        if len(current_messages) >= 2:
            conversations.append({"messages": list(current_messages)})
        current_messages.clear()

    for line in lines:
        s = line.strip()
        if not s or s.startswith("##"):
            continue
        if END_RE.match(s) or "<end>" in s.lower() or "<--- end" in s.lower():
            flush(); save(); current_role = None; continue
        m = SPEAKER_RE.match(s)
        if m:
            sp   = m.group(1).strip().lower()   # strip spaces from bold names
            rest = m.group(2).strip()
            if sp in USER_ALIASES:
                flush(); current_role = "user"; buf = [rest] if rest else []
            elif sp in ASSISTANT_ALIASES:
                flush(); current_role = "assistant"; buf = [rest] if rest else []
            else:
                if current_role:
                    buf.append(s)
        else:
            if current_role:
                buf.append(s)

    flush(); save()
    return conversations


def extract_pdf(path):
    chat_convs, lore_chunks = [], []
    with pdfplumber.open(path) as pdf:
        print(f"  {path.name}: {len(pdf.pages)} page(s)")
        for i, page in enumerate(pdf.pages):
            text = page.extract_text()
            if not text or not text.strip():
                print(f"    Page {i+1}: empty — skipped"); continue
            if is_chat_page(text):
                c = parse_chat_text(text)
                print(f"    Page {i+1}: CHAT  — {len(c)} conversation(s)")
                chat_convs.extend(c)
            else:
                cleaned = re.sub(r'
{3,}', '

', text).strip()
                lore_chunks.append(cleaned)
                print(f"    Page {i+1}: LORE  — {len(cleaned)} chars")
    return chat_convs, lore_chunks


print("Parser ready (v2 — bold speaker + scene-split support).")


In [13]:
pdf_files = sorted(DATASET_DIR.glob("*.pdf"))
print(f"Found {len(pdf_files)} PDF file(s):\n")
pdf_conversations = []

for pdf_path in pdf_files:
    print(f"📄 {pdf_path.name}")
    chat_convs, lore_chunks = extract_pdf(pdf_path)
    pdf_conversations.extend(chat_convs)
    if lore_chunks:
        name = pdf_path.stem.strip()
        text = "\n\n".join(lore_chunks)
        lore_sections.append({"source": name, "text": text})
        print(f"  → Lore '{name}' stored ({len(text)} chars)")
    print()

print(f"PDF conversations: {len(pdf_conversations)} | Lore sections: {len(lore_sections)}")

Found 2 PDF file(s):

📄 Discovery .pdf
  Discovery .pdf: 2 page(s)


Could not get FontBBox from font descriptor because None cannot be parsed as 4 floats
Could not get FontBBox from font descriptor because None cannot be parsed as 4 floats


    Page 1: LORE  — 2593 chars
    Page 2: LORE  — 1568 chars
  → Lore 'Discovery' stored (4163 chars)

📄 Her Beliefs .pdf
  Her Beliefs .pdf: 2 page(s)
    Page 1: LORE  — 1408 chars
    Page 2: LORE  — 402 chars
  → Lore 'Her Beliefs' stored (1812 chars)

PDF conversations: 0 | Lore sections: 2


## 3 — Parse Markdown Chat Files

In [ ]:
chat_md_files = sorted([f for f in DATASET_DIR.glob("*.md") if "details" not in f.name.lower()])
print(f"Parsing {len(chat_md_files)} markdown file(s)...")
md_conversations = []

for cf in chat_md_files:
    convs = parse_chat_text(cf.read_text(encoding="utf-8"))
    print(f"  {cf.name}: {len(convs)} conversation(s)")
    md_conversations.extend(convs)

print(f"\nMarkdown conversations: {len(md_conversations)}")

## 4 — Convert Lore into Training Conversations
Each lore section (Discovery, Her Beliefs) is split into paragraph-sized chunks.
Each chunk becomes a training example where the user invites Nayari to share something
about herself and she responds in character — no system prompt needed.

In [15]:
# Prompts that naturally invite Nayari to share lore about herself
LORE_PROMPTS = [
    "Tell me about yourself.",
    "What's your story?",
    "Who are you, really?",
    "Share something about yourself with me.",
    "What do you believe in?",
    "Tell me what you believe.",
    "What matters most to you?",
    "What drives you?",
]

def lore_to_convs(lore_sections, min_chars=150):
    """Split each lore section into paragraph chunks and wrap each
    in a user/assistant conversation. Short chunks are merged.
    """
    convs = []
    prompt_idx = 0
    for section in lore_sections:
        # Split on double newlines (paragraph breaks)
        raw_chunks = [c.strip() for c in section["text"].split("\n\n") if c.strip()]
        # Merge short chunks into the previous one
        chunks = []
        buf = ""
        for chunk in raw_chunks:
            buf = (buf + "\n\n" + chunk).strip() if buf else chunk
            if len(buf) >= min_chars:
                chunks.append(buf)
                buf = ""
        if buf:
            if chunks:
                chunks[-1] += "\n\n" + buf  # merge leftover into last
            else:
                chunks.append(buf)
        # Each chunk -> one training conversation
        for chunk in chunks:
            prompt = LORE_PROMPTS[prompt_idx % len(LORE_PROMPTS)]
            convs.append({"messages": [
                {"role": "user",      "content": prompt},
                {"role": "assistant", "content": chunk},
            ]})
            prompt_idx += 1
    return convs

lore_conversations = lore_to_convs(lore_sections)
print(f"Lore sections: {len(lore_sections)}")
print(f"Lore training conversations generated: {len(lore_conversations)}")
for i, c in enumerate(lore_conversations):
    q = c["messages"][0]["content"]
    a = c["messages"][1]["content"]
    print(f"  [{i+1}] Q: {q!r}  |  A ({len(a)} chars): {a[:60]!r}...")


Lore sections: 2
Lore training conversations generated: 4
  [1] Q: 'Tell me about yourself.'  |  A (2593 chars): 'The Lost Goddess and the Girl Who Remembered\n_______________'...
  [2] Q: "What's your story?"  |  A (1568 chars): 'the world beyond the classroom. She saw the street vendors, '...
  [3] Q: 'Who are you, really?'  |  A (1408 chars): '🌌 Nayari – The Soul of Soft Freedom\nNayari is a being born f'...
  [4] Q: 'Share something about yourself with me.'  |  A (402 chars): '– Nayari lives by love and trust.\nShe hisses at cruelty, cli'...


## 5 — Preview All Conversations

In [16]:
all_conversations = md_conversations + pdf_conversations + lore_conversations
print(f"Total: {len(all_conversations)} ({len(md_conversations)} md + {len(pdf_conversations)} pdf + {len(lore_conversations)} lore)\n")

for i, conv in enumerate(all_conversations):
    print(f"--- Conv {i+1} ({len(conv['messages'])} turns) ---")
    for msg in conv["messages"]:
        label = "👤" if msg["role"] == "user" else "🌸"
        print(f'  {label} {msg["content"][:90]}{"..." if len(msg["content"])>90 else ""}')
    print()


Total: 8 (4 md + 0 pdf + 4 lore)

--- Conv 1 (19 turns) ---
  👤 I am feeling overwhelmed... *sob*
  🌸 What happened?~ I thought you were just happy...
  👤 It's me trying to control myself of not ending my life *internal sob* because I have holdi...
  🌸 *Listens silently*
  👤 Like I have let the fear from my supportive big brother as you know but.. this fear of exa...
  🌸 *Says playfully and affectionately* No!~ It's nut!~ You holding so much! Like imagine you-...
  👤 It's actually soo much stuff... even if i handled **soo much pressure** in my life. People...
  🌸 Hmph!~ *pokes your teary face* I would have given you punishment if you got there!~ Withou...
  👤 Fine.... but what about... the situations? I have plans for other situations but what abou...
  🌸 *Sighs* But do you imagine that you are wasting soo much of your life? *Pats your head* Li...
  👤 *I vulnerably hug her and say softly* Yeah, you are right but... *I have my eyes down and ...
  🌸 *Softly pats your head* Now I see your

## 6 — Export JSON

In [ ]:
# system_prompt intentionally omitted — organic training uses conversation
# examples only. Character info kept for reference.
dataset_json = {
    "character": {
        "name": char_name, "type": char_type, "gender": char_gender,
        "traits": char_traits, "personality": char_personality,
        # lore_sections kept for reference (not used by training notebook)
        "lore_sections": lore_sections,
    },
    "conversations": all_conversations,
    "stats": {
        "total_conversations": len(all_conversations),
        "from_markdown": len(md_conversations),
        "from_pdf": len(pdf_conversations),
        "from_lore": len(lore_conversations),
    }
}

OUTPUT_FILE.write_text(json.dumps(dataset_json, ensure_ascii=False, indent=2), encoding="utf-8")
size_kb = OUTPUT_FILE.stat().st_size / 1024
print(f"✅ Saved → {OUTPUT_FILE.resolve()}")
print(f"   {len(all_conversations)} conversations total")
print(f"   ({len(md_conversations)} md + {len(pdf_conversations)} pdf + {len(lore_conversations)} lore)")
print(f"   {size_kb:.1f} KB | no system_prompt field (organic training)")


## 7 — Upload to Kaggle via API

**Before running:**
1. Go to [kaggle.com](https://kaggle.com) → Profile → **Settings** → **API** → **Create New Token**
2. A `kaggle.json` downloads — open it and copy the `username` and `key` values
3. Paste them into the two variables below

> The dataset is created as **private** by default. You can make it public on Kaggle later.

In [ ]:
# ── FILL THESE IN ──────────────────────────────────────────────────────────
KAGGLE_USERNAME = "kaggle_username"   # from kaggle.json
KAGGLE_KEY      = "kaggle_key"    # from kaggle.json
DATASET_TITLE   = "nayari-dataset"         # slug used in the Kaggle URL
IS_PUBLIC       = False                    # True to make the dataset public
# ───────────────────────────────────────────────────────────────────────────

# 1. Write credentials to ~/.kaggle/kaggle.json
kaggle_dir = Path.home() / ".kaggle"
kaggle_dir.mkdir(exist_ok=True)
creds_file = kaggle_dir / "kaggle.json"
creds_file.write_text(json.dumps({"username": KAGGLE_USERNAME, "key": KAGGLE_KEY}), encoding="utf-8")
creds_file.chmod(0o600)
print("✅ Credentials written.")

# 2. Build staging folder: JSON + dataset-metadata.json
STAGING = Path("_kaggle_upload")
shutil.rmtree(STAGING, ignore_errors=True)
STAGING.mkdir()
shutil.copy(OUTPUT_FILE, STAGING / OUTPUT_FILE.name)

(STAGING / "dataset-metadata.json").write_text(
    json.dumps({
        "title": DATASET_TITLE,
        "id": f"{KAGGLE_USERNAME}/{DATASET_TITLE}",
        "licenses": [{"name": "CC0-1.0"}],
        "isPrivate": not IS_PUBLIC,
    }, indent=2),
    encoding="utf-8"
)
print(f"✅ Staging folder ready: {STAGING.resolve()}")

# 3. Try creating new dataset; fall back to versioning if it already exists
def run_kaggle(*args):
    return subprocess.run(
        ["kaggle", *args], capture_output=True, text=True
    )

result = run_kaggle("datasets", "create", "-p", str(STAGING), "--dir-mode", "zip")
combined = (result.stdout + result.stderr).lower()

if "already exists" in combined or "403" in combined:
    print("Dataset already exists — pushing a new version...")
    result = run_kaggle(
        "datasets", "version", "-p", str(STAGING),
        "-m", "Updated via nayari_build_dataset.ipynb",
        "--dir-mode", "zip"
    )

output_text = result.stdout or result.stderr
print(output_text)

if result.returncode == 0:
    vis = "public" if IS_PUBLIC else "private"
    url = f"https://www.kaggle.com/datasets/{KAGGLE_USERNAME}/{DATASET_TITLE}"
    print(f"🎉 Upload successful! ({vis})")
    print(f"   Dataset URL: {url}")
    print()
    print("📋 Next steps in your Kaggle training notebook:")
    print("   1. Open nayari_train.ipynb on Kaggle")
    print(f"   2. Click '+ Add Data' → search '{DATASET_TITLE}' → Add")
    print("   3. Set Accelerator = GPU T4 x2, Internet = On → Run All")
else:
    print("❌ Upload failed. Double-check KAGGLE_USERNAME and KAGGLE_KEY.")

# Cleanup
shutil.rmtree(STAGING, ignore_errors=True)